
**Project question:** which Sicily listings have unusual prices, and are those prices explained by size and amenities?

| Part | What it does | Why |
|---|---|---|
| 1. Load | Rebuilds `df_clean` exactly as the cleaning log describes and checks the log's numbers | Makes this notebook run on its own |
| 2. Filtering | Picks subsets of rows (e.g. flagged listings) | Focus on listings that need a closer look |
| 3. Group-by + aggregation | Splits listings into comparable groups and computes counts, medians, rates | Shows *where* unusual prices concentrate and compares like with like |
| 4. Sensitivity | Repeats key results with and without flagged listings | The log promises this check |
| 5. Save | Writes every table to `outputs/tables/` | Output for the next steps |

**Reading guide:** the price measure is *price per guest per night* (`price / accommodates`), same as the cleaning log. Medians are used instead of means because prices are right-skewed.

In [1]:
import os, re, json
from pathlib import Path
import numpy as np
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

ROOT = Path(os.environ.get("PROJECT_ROOT", ".."))                       # repo root (notebook sits in notebooks/)
RAW_PATH = Path(os.environ.get("SICILY_CSV", ROOT / "data" / "raw" / "Sicily.csv"))
OUT_DIR = ROOT / "outputs" / "tables"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("raw file:", RAW_PATH, "| exists:", RAW_PATH.exists())

raw file: ../data/raw/Sicily.csv | exists: True


## 1. Load data and rebuild `df_clean`

The next cell repeats the cleaning log step by step. The `assert` lines compare the result with the numbers written in the log, so any difference stops the notebook instead of passing silently.

> If you already have `df_clean` from the cleaning notebook, you can skip this cell and load your own file instead. Keep the asserts to check it.

In [2]:
KEEP = ["id", "neighbourhood_cleansed", "latitude", "longitude", "property_type", "room_type", "accommodates",
        "bathrooms_text", "bedrooms", "amenities", "price", "minimum_nights", "number_of_reviews",
        "review_scores_rating"]                                           # the 14 columns kept (column selection)
raw = pd.read_csv(RAW_PATH, usecols=KEEP)
assert len(raw) == 56873 and raw["id"].is_unique                          # steps 1-2

df = raw.copy()
df["price"] = pd.to_numeric(df["price"].astype(str).str.replace(r"[\$,]", "", regex=True), errors="coerce")   # step 2
df = df[df["price"].notna()].copy()                                       # step 3
assert len(df) == 51864

def bath(t):                                                              # step 4
    if pd.isna(t): return np.nan
    m = re.search(r"\d+(\.\d+)?", t)
    return float(m.group()) if m else 0.5                                  # "Half-bath" -> 0.5
df["bathrooms"] = df["bathrooms_text"].apply(bath)
df["amenities_count"] = df["amenities"].apply(lambda s: len(json.loads(s)))   # step 5
df.loc[(df["review_scores_rating"] < 1) | (df["review_scores_rating"] > 5), "review_scores_rating"] = np.nan  # step 7
df["minimum_nights_flag"] = df["minimum_nights"] > 365                    # step 10

ppg = df["price"] / df["accommodates"]                                    # steps 12-14 use log10 of price per guest
lg = np.log10(ppg)
q1, q3 = lg.quantile([0.25, 0.75]); iqr = q3 - q1
high_fence, low_fence = 10 ** (q3 + 3 * iqr), 10 ** (q1 - 3 * iqr)
df["high_price_per_guest_flag"] = ppg > high_fence                        # step 12
df["low_price_per_guest_flag"] = ppg < low_fence                          # step 13
g = lg.groupby(df["room_type"]); a, b = g.quantile(0.25), g.quantile(0.75)
room_fence = 10 ** (b + 1.5 * (b - a))
df["price_outlier"] = ppg > df["room_type"].map(room_fence)               # step 14

df_clean = df.copy()
assert df_clean.shape == (51864, 20)
assert round(high_fence, 2) == 507.97 and round(low_fence, 2) == 2.58
assert df_clean["high_price_per_guest_flag"].sum() == 40
assert df_clean["low_price_per_guest_flag"].sum() == 14
assert df_clean["price_outlier"].sum() == 669
print("df_clean matches the cleaning log:", df_clean.shape)
print("room-type fences (USD per guest per night):", room_fence.round(2).to_dict())

df_clean matches the cleaning log: (51864, 20)
room-type fences (USD per guest per night): {'Entire home/apt': 163.28, 'Hotel room': 167.3, 'Private room': 161.32, 'Shared room': 179.27}


### Helper columns for the analysis

These are *working columns* for this step only, not part of the cleaning log.

- `price_per_guest` — main price measure.
- `bedrooms_grp` — bedrooms capped at 4 (4 means "4 or more"). `NaN` stays `NaN`.
- `peer_median_amenities` — median `amenities_count` of **peers** (same room type and same `bedrooms_grp`). Used to ask whether a high price comes with more amenities than similar listings.
- `any_flag` — listing carries at least one flag (high, low or room-type outlier).

In [3]:
d = df_clean.copy()
d["price_per_guest"] = d["price"] / d["accommodates"]
d["bedrooms_grp"] = d["bedrooms"].clip(upper=4)
d["peer_median_amenities"] = d.groupby(["room_type", "bedrooms_grp"])["amenities_count"].transform("median")
d["amenities_vs_peers"] = d["amenities_count"] - d["peer_median_amenities"]
d["reviews_bucket"] = pd.cut(d["number_of_reviews"], [-1, 0, 9, 10**6], labels=["0", "1-9", "10+"])
d["any_flag"] = d["price_outlier"] | d["high_price_per_guest_flag"] | d["low_price_per_guest_flag"]
print(d[["price_outlier", "high_price_per_guest_flag", "low_price_per_guest_flag", "any_flag"]].sum())
print("bedrooms missing (no peer group):", int(d["bedrooms"].isna().sum()))

price_outlier                669
high_price_per_guest_flag     40
low_price_per_guest_flag      14
any_flag                     683
dtype: int64
bedrooms missing (no peer group): 7859


## 2. Filtering

A filter keeps only the rows that satisfy a condition. All filters below start from the flagged listings, because that is where the project question lives. They **overlap** (one listing can be in several), so their row counts must not be added up.

In [4]:
SHOW = ["id", "neighbourhood_cleansed", "property_type", "room_type", "accommodates", "bedrooms",
        "amenities_count", "peer_median_amenities", "number_of_reviews", "review_scores_rating",
        "price", "price_per_guest"]
out = d[d["price_outlier"]]

filters = {
    "F1_outliers":     (out,                                                                    "price_outlier = True (above room-type fence, step 14)"),
    "F2_high_flag":    (d[d["high_price_per_guest_flag"]],                                      "far-out high price per guest (step 12)"),
    "F3_low_flag":     (d[d["low_price_per_guest_flag"]],                                       "far-out low price per guest (step 13)"),
    "F4_not_explained":(out[out["amenities_vs_peers"] <= 0],                                    "outlier AND amenities <= median of peers: price not explained by amenities"),
    "F5_credible":     (out[(out["number_of_reviews"] >= 10) & (out["review_scores_rating"] >= 4.5)], "outlier AND >=10 reviews AND rating >=4.5: likely a genuine premium listing"),
    "F6_no_reviews":   (out[out["number_of_reviews"] == 0],                                     "outlier AND zero reviews: no guest evidence for the price"),
    "F7_top20":        (out.nlargest(20, "price_per_guest"),                                    "20 highest prices per guest among outliers"),
}
filter_summary = pd.DataFrame([(k, v[1], len(v[0]), round(len(v[0]) / len(d) * 100, 2)) for k, v in filters.items()],
                              columns=["filter", "condition", "rows", "% of df_clean"])
filter_summary

,filter,condition,rows,% of df_clean
0,F1_outliers,"price_outlier = True (above room-type fence, s...",669,1.29
1,F2_high_flag,far-out high price per guest (step 12),40,0.08
2,F3_low_flag,far-out low price per guest (step 13),14,0.03
3,F4_not_explained,outlier AND amenities <= median of peers: pric...,364,0.70
4,F5_credible,outlier AND >=10 reviews AND rating >=4.5: lik...,108,0.21
5,F6_no_reviews,outlier AND zero reviews: no guest evidence fo...,333,0.64
6,F7_top20,20 highest prices per guest among outliers,20,0.04


In [5]:
filters["F7_top20"][0][SHOW].round(2)

,id,neighbourhood_cleansed,property_type,room_type,accommodates,bedrooms,amenities_count,peer_median_amenities,number_of_reviews,review_scores_rating,price,price_per_guest
26112,823851890735056903,San Vito Lo Capo,Room in hotel,Private room,1,1.0,27,22.0,0,NaN,61160.55,61160.55
20149,52250690,Modica,Entire home,Entire home/apt,4,2.0,44,35.0,0,NaN,88494.55,22123.64
41285,1332491092213117150,Trapani,Entire condo,Entire home/apt,2,1.0,64,33.0,59,4.86,9128.51,4564.26
8113,20017976,Ravanusa,Private room in bed and breakfast,Private room,4,2.0,26,22.0,1,5.00,17243.16,4310.79
29728,930598030535216768,Capo d'Orlando,Entire condo,Entire home/apt,5,2.0,39,35.0,0,NaN,16560.67,3312.13
34683,1134674862349280891,Giardini-Naxos,Entire condo,Entire home/apt,4,1.0,57,33.0,51,4.94,11447.00,2861.75
16061,43288399,Catania,Entire loft,Entire home/apt,2,NaN,12,NaN,1,5.00,5584.00,2792.00
44554,1410711849182017985,Palermo,Entire rental unit,Entire home/apt,2,1.0,6,33.0,0,NaN,3424.00,1712.00
33684,1111657263416400295,Marsala,Entire rental unit,Entire home/apt,4,2.0,30,35.0,0,NaN,5988.00,1497.00
15275,40395372,Avola,Entire villa,Entire home/apt,4,2.0,31,35.0,4,5.00,5670.00,1417.50


## 3. Group-by and aggregation

*Group-by* splits the rows into groups (by room type, area, number of bedrooms...). *Aggregation* turns each group into a few numbers. The same `stats()` function is used everywhere so all tables have the same columns and can be compared.

Column meaning: `listings` = rows in the group; `*_ppg` = price per guest per night; `outliers` / `outlier_rate_%` = listings with `price_outlier = True`.

In [6]:
def stats(g):
    return pd.Series({
        "listings": len(g),
        "median_price": g["price"].median(),
        "median_ppg": g["price_per_guest"].median(),
        "mean_ppg": g["price_per_guest"].mean(),
        "std_ppg": g["price_per_guest"].std(),
        "min_ppg": g["price_per_guest"].min(),
        "max_ppg": g["price_per_guest"].max(),
        "median_amenities": g["amenities_count"].median(),
        "outliers": int(g["price_outlier"].sum()),
        "outlier_rate_%": g["price_outlier"].mean() * 100,
    })
def by(df, key):
    return df.groupby(key, observed=True).apply(stats, include_groups=False).reset_index()

### G1 — By room type
How do the price levels and the outlier fences differ between room types? (Hotel room and Shared room have few listings: read them with care.)

In [7]:
G1 = by(d, "room_type")
G1.insert(2, "outlier_fence_ppg", G1["room_type"].map(room_fence))
G1.round(2)

,room_type,listings,outlier_fence_ppg,median_price,median_ppg,mean_ppg,std_ppg,min_ppg,max_ppg,median_amenities,outliers,outlier_rate_%
0,Entire home/apt,41131.0,163.28,137.00,34.91,44.99,120.93,0.34,22123.64,34.0,554.0,1.35
1,Hotel room,178.0,167.30,118.76,43.35,52.70,30.83,10.00,193.80,21.0,3.0,1.69
2,Private room,10495.0,161.32,95.00,39.44,52.46,599.14,1.96,61160.55,22.0,112.0,1.07
3,Shared room,60.0,179.27,33.00,30.00,26.48,14.67,2.50,76.50,21.0,0.0,0.00


### G2 — By area (`neighbourhood_cleansed`)
Only areas with at least 100 listings, sorted by outlier rate. With fewer listings the rate jumps around too much to be meaningful.

In [8]:
G2 = by(d, "neighbourhood_cleansed")
G2 = G2[G2["listings"] >= 100].sort_values("outlier_rate_%", ascending=False).reset_index(drop=True)
print(len(G2), "areas with >= 100 listings")
G2.head(15).round(2)

74 areas with >= 100 listings


,neighbourhood_cleansed,listings,median_price,median_ppg,mean_ppg,std_ppg,min_ppg,max_ppg,median_amenities,outliers,outlier_rate_%
0,Taormina,1225.0,199.13,63.75,75.39,51.19,4.30,489.14,34.0,69.0,5.63
1,Pantelleria,506.0,184.50,55.32,66.76,46.07,3.34,368.44,29.0,20.0,3.95
2,Santa Marina Salina,103.0,220.63,56.32,70.24,45.93,7.61,320.00,19.0,4.0,3.88
3,Letojanni,239.0,140.90,36.50,50.49,54.29,8.41,403.07,33.0,9.0,3.77
4,Cefal,907.0,188.00,57.12,70.97,66.25,9.43,900.00,35.0,34.0,3.75
5,Partinico,194.0,192.50,40.16,55.42,45.89,8.21,375.00,37.0,7.0,3.61
6,Trecastagni,126.0,141.91,34.58,54.41,100.74,9.69,1058.00,23.0,4.0,3.17
7,Avola,912.0,137.00,36.26,49.68,62.06,2.95,1417.50,33.0,26.0,2.85
8,Ragusa,1223.0,141.67,38.58,49.70,51.84,7.00,1106.18,29.0,31.0,2.53
9,Riposto,179.0,123.67,28.60,39.94,44.52,4.66,355.00,40.0,4.0,2.23


### G3 — By number of bedrooms (entire homes)
Size is the main driver of price. Comparing listings only inside the same size group avoids calling a big villa "unusual" just because it is big. Listings with unknown bedrooms are left out.

In [9]:
eh = d[(d["room_type"] == "Entire home/apt") & d["bedrooms_grp"].notna()].copy()
G3 = by(eh, "bedrooms_grp")
G3["bedrooms_grp"] = G3["bedrooms_grp"].map(lambda x: "4+" if x >= 4 else str(int(x)))
G3.round(2)

,bedrooms_grp,listings,median_price,median_ppg,mean_ppg,std_ppg,min_ppg,max_ppg,median_amenities,outliers,outlier_rate_%
0,1,18007.0,111.50,35.89,44.02,53.50,2.66,4564.26,33.0,169.0,0.94
1,2,14098.0,146.67,32.00,41.29,191.74,0.34,22123.64,35.0,136.0,0.96
2,3,5289.0,221.00,34.22,45.99,43.97,0.92,1104.68,38.0,87.0,1.64
3,4+,2821.0,506.80,51.50,66.17,60.85,0.81,875.00,37.0,150.0,5.32


### G4 — Bedrooms × amenities tier (entire homes)
Inside each bedroom group, listings are split into low / mid / high thirds of `amenities_count`. Question: for the same size, do listings with more amenities cost more, and are they flagged more often? (`amenities_count` measures quantity, not quality.)

In [10]:
eh["amenities_tier"] = eh.groupby("bedrooms_grp")["amenities_count"].transform(
    lambda s: pd.qcut(s, 3, labels=["low", "mid", "high"], duplicates="drop"))
G4 = by(eh, ["bedrooms_grp", "amenities_tier"])
G4["bedrooms_grp"] = G4["bedrooms_grp"].map(lambda x: "4+" if x >= 4 else str(int(x)))
G4.round(2)

,bedrooms_grp,amenities_tier,listings,median_price,median_ppg,mean_ppg,std_ppg,min_ppg,max_ppg,median_amenities,outliers,outlier_rate_%
0,1,low,6297.0,107.33,35.50,43.23,39.52,2.66,1712.00,16.0,57.0,0.91
1,1,mid,5936.0,109.00,35.33,42.72,31.35,2.77,431.62,34.0,57.0,0.96
2,1,high,5774.0,119.00,36.98,46.23,78.78,4.96,4564.26,48.0,55.0,0.95
3,2,low,4952.0,136.97,30.06,37.49,31.77,0.34,856.00,16.0,42.0,0.85
4,2,mid,4502.0,143.00,31.50,40.21,67.31,1.67,3312.13,35.0,46.0,1.02
5,2,high,4644.0,161.67,34.25,46.38,325.76,3.54,22123.64,51.0,48.0,1.03
6,3,low,1765.0,199.00,31.64,44.54,44.03,1.01,625.75,19.0,37.0,2.10
7,3,mid,1773.0,205.50,32.81,44.23,42.17,0.92,687.34,38.0,30.0,1.69
8,3,high,1751.0,252.40,38.75,49.24,45.50,3.04,1104.68,56.0,20.0,1.14
9,4+,low,974.0,470.40,50.25,69.80,74.03,0.81,875.00,21.0,69.0,7.08


### G5 — By property type (top 10 by number of listings)

In [11]:
top_pt = d["property_type"].value_counts().head(10).index
G5 = by(d[d["property_type"].isin(top_pt)], "property_type").sort_values("listings", ascending=False).reset_index(drop=True)
G5.round(2)

,property_type,listings,median_price,median_ppg,mean_ppg,std_ppg,min_ppg,max_ppg,median_amenities,outliers,outlier_rate_%
0,Entire rental unit,16662.0,125.75,33.38,40.46,37.28,0.34,1712.00,31.0,125.0,0.75
1,Entire home,9783.0,142.67,34.50,45.87,226.02,2.72,22123.64,32.0,111.0,1.13
2,Private room in bed and breakfast,4382.0,93.76,39.38,45.62,74.30,1.96,4310.79,19.0,27.0,0.62
3,Entire condo,4347.0,124.00,32.52,41.05,100.07,2.77,4564.26,40.0,27.0,0.62
4,Entire villa,4074.0,351.83,53.96,69.22,65.70,1.01,1417.50,42.0,218.0,5.35
5,Entire vacation home,3450.0,127.67,32.93,38.73,26.56,3.33,500.00,42.0,22.0,0.64
6,Private room in rental unit,2236.0,91.29,37.63,43.02,28.67,2.42,677.87,23.0,8.0,0.36
7,Private room in home,1160.0,88.08,35.50,41.31,27.21,2.60,276.55,21.0,4.0,0.34
8,Private room in condo,573.0,84.67,35.67,44.21,42.67,4.36,474.00,32.0,10.0,1.75
9,Entire loft,365.0,109.56,32.50,45.72,145.97,4.96,2792.00,36.0,2.0,0.55


### G6 — By number of reviews
Are listings with a high price more likely to have no reviews? (Zero reviews = no guest has confirmed the price is accepted.)

In [12]:
G6 = by(d, "reviews_bucket").rename(columns={"reviews_bucket": "number_of_reviews"})
G6.round(2)

,number_of_reviews,listings,median_price,median_ppg,mean_ppg,std_ppg,min_ppg,max_ppg,median_amenities,outliers,outlier_rate_%
0,0,14499.0,132.38,39.00,55.98,542.51,0.34,61160.55,20.0,333.0,2.30
1,1-9,18023.0,126.00,35.56,44.40,54.64,0.43,4310.79,31.0,225.0,1.25
2,10+,19342.0,122.00,34.17,41.38,48.97,1.67,4564.26,39.0,111.0,0.57


## 4. Sensitivity check — with and without flagged listings

The cleaning log says all flagged listings are kept and that the analysis is run with and without them. If medians and rankings barely move, the conclusions do not depend on the flagged listings.

In [13]:
clean_only = d[~d["any_flag"]]
G7 = (by(d, "room_type")[["room_type", "listings", "median_ppg", "mean_ppg"]]
      .merge(by(clean_only, "room_type")[["room_type", "listings", "median_ppg", "mean_ppg"]],
             on="room_type", suffixes=("_all", "_without_flags")))
G7["median_change_%"] = (G7["median_ppg_without_flags"] / G7["median_ppg_all"] - 1) * 100
G7["mean_change_%"] = (G7["mean_ppg_without_flags"] / G7["mean_ppg_all"] - 1) * 100
display(G7.round(2))

# do areas keep the same ranking by median price per guest?
a_all = d.groupby("neighbourhood_cleansed")["price_per_guest"].agg(["size", "median"])
a_wo = clean_only.groupby("neighbourhood_cleansed")["price_per_guest"].median()
cmp = a_all[a_all["size"] >= 100].join(a_wo.rename("median_without_flags"), how="inner")
rho = cmp["median"].corr(cmp["median_without_flags"], method="spearman")
print(f"Spearman rank correlation of area medians, with vs without flags: {rho:.3f} ({len(cmp)} areas)")

,room_type,listings_all,median_ppg_all,mean_ppg_all,listings_without_flags,median_ppg_without_flags,mean_ppg_without_flags,median_change_%,mean_change_%
0,Entire home/apt,41131.0,34.91,44.99,40570.0,34.62,41.18,-0.81,-8.48
1,Hotel room,178.0,43.35,52.70,175.0,43.00,50.55,-0.82,-4.07
2,Private room,10495.0,39.44,52.46,10377.0,39.00,43.77,-1.10,-16.56
3,Shared room,60.0,30.00,26.48,59.0,30.00,26.88,0.00,1.53


Spearman rank correlation of area medians, with vs without flags: 0.995 (74 areas)


## 5. Key takeaways (from this run)

- **How many are flagged:** 669 listings (1.29%) are `price_outlier`; 40 of them are far-out high and 14 listings are far-out low.
- **Size still matters, but much less than before:** the outlier rate is 0.94% for 1 bedroom and 5.32% for 4+ bedrooms (G3); entire villas have 5.35% (G5). Flags are now spread more evenly across sizes because they use price per guest.
- **Amenities do not explain the flagged prices:** in the same bedroom group, more amenities go with a slightly higher median price (3 bedrooms: 199 → 252 USD per night) but not with more flags; the outlier rate actually falls (3 bedrooms: 2.10% → 1.14%; 4+: 7.08% → 3.34%) (G4). 364 of the outliers (F4) have no more amenities than their peers.
- **Reviews:** half of the outliers (333 of 669, F6) have zero reviews, while only 28% of all listings have none. The outlier rate is 2.30% with 0 reviews and 0.57% with 10+ reviews (G6). Only 108 outliers (F5) combine 10+ reviews with rating ≥ 4.5, so they are the most credible premium listings.
- **Areas (G2):** Taormina has the highest outlier rate (5.63%, 69 of 1,225 listings), followed by Pantelleria (3.95%), Santa Marina Salina (3.88%), Letojanni (3.77%) and Cefal (3.75%), against 1.29% overall.
- **Sensitivity (G7):** removing the 683 flagged listings changes the median price per guest by about 1% or less in every room type, and the ranking of areas stays almost identical (Spearman 0.995). The mean changes much more (−8.5% for entire homes, −16.6% for private rooms), so **medians should be used for conclusions**.

**Limits to remember**

- 7,859 listings (15%) have no bedroom count, so they are missing from G3, G4 and F4.
- `amenities_count` measures quantity, not quality.
- A flag only says a price is unusual within its room type; it does not prove the price is wrong or the listing is overpriced.
- Hotel room (178) and Shared room (60) are small groups.
- The area name "Cefal" is written like this in the raw data (probably Cefalù cut off); it was not changed because the cleaning log does not cover it.

## 6. Save outputs

Every table is written as CSV (small, readable in git diffs) to `outputs/tables/`.

In [14]:
tables = {name: v[0][SHOW] for name, v in filters.items()}
tables.update({"F0_filter_summary": filter_summary, "G1_room_type": G1, "G2_area": G2, "G3_bedrooms": G3,
               "G4_bedrooms_amenities": G4, "G5_property_type": G5, "G6_reviews": G6, "G7_sensitivity": G7})
for name, t in tables.items():
    t.round(2).to_csv(OUT_DIR / f"{name}.csv", index=False)
print(len(tables), "files written to", OUT_DIR.resolve())
sorted(p.name for p in OUT_DIR.glob("*.csv"))

15 files written to /home/claude/repo/outputs/tables


['F0_filter_summary.csv',
 'F1_outliers.csv',
 'F2_high_flag.csv',
 'F3_low_flag.csv',
 'F4_not_explained.csv',
 'F5_credible.csv',
 'F6_no_reviews.csv',
 'F7_top20.csv',
 'G1_room_type.csv',
 'G2_area.csv',
 'G3_bedrooms.csv',
 'G4_bedrooms_amenities.csv',
 'G5_property_type.csv',
 'G6_reviews.csv',
 'G7_sensitivity.csv']